In [57]:
import os
import yaml
import json
import mne
import os
import numpy as np
import pandas as pd
import random
from datetime import timedelta
import matplotlib
import matplotlib.pyplot  as plt
import matplotlib.dates as mdates
import PyQt5
matplotlib.use('Qt5Agg')



### Functions

In [77]:
def read_file(eeg_path):
    fname, extension = os.path.splitext(eeg_path)
    if extension.lower() == '.fif':
        raw = mne.io.read_raw_fif(eeg_path, preload=True)
    elif extension.lower() == '.edf':
        raw = mne.io.read_raw_edf(eeg_path, preload=True)
    else:
        raise ValueError(f'File extension {extension} not supported')
    return fname, raw

def get_dc_channels(raw, threshold):
    # Get the channel names and pick only DC channels
    dc_channels = [ch for ch in raw.ch_names if 'DC' in ch]
    dc_picks = mne.pick_channels(raw.ch_names, include=dc_channels)

    signal_channels = []
    # Check which DC channels exceed the threshold
    for idx in dc_picks:
        data = raw.get_data(picks=idx)  # Extract data for the channel
        if np.any(data > threshold):  # Check if any value exceeds the threshold
            signal_channels.append(raw.ch_names[idx])
    print(f"DC channels with signals: {signal_channels}")
    return signal_channels

def load_eeg(eeg_path, config):
    """
    Load EEG data from a file and apply preprocessing steps
    param:
        eeg_path: Path to the EEG file
        config: Configuration dictionary
    return:
        fname: File name
        raw: Raw EEG data
    """
    # Load EEG data    
    fname, raw = read_file(eeg_path)
    print(f"channels: {raw.info['ch_names']}")

    # Resample data
    if config.get('sfreq', False):
        print(f"Resampling data to {config['sfreq']} Hz")
        raw.resample(config['sfreq'])

    # Filter data
    lo_pass = config.get('l_freq', None)
    hi_pass = config.get('h_freq', None)    
    raw.filter(lo_pass, hi_pass)

    # Rename channels
    raw= raw.rename_channels(config['channel_map'])

    # Change DC channels types
    channel_type_mapping = {ch: 'misc' for ch in raw.info['ch_names'] if "DC" in ch}
    raw.set_channel_types(channel_type_mapping)

    # Pick channels
    missing_channels = [x for x in config['channels'] if x not in raw.info['ch_names']]
    if len(missing_channels) > 0:
        raise ValueError(f"Missing channels: {missing_channels}")
    channels = config['channels']
    dc_channel = get_dc_channels(raw, config['dc_threshold'])
    channels.extend(dc_channel)
    channels = list(set(channels))
    raw.pick(channels)
    return fname, raw, dc_channel

def get_eeg_timestamps(raw_data):
    # Start time of the recording
    start_time = raw_data.info['meas_date']

    # Duration of the recording
    n_samples = raw_data.n_times  # Number of samples
    sampling_frequency = raw_data.info['sfreq']  # Sampling frequency
    duration = timedelta(seconds=n_samples / sampling_frequency)
    # End time of the recording
    end_time = start_time + duration

    # Adjust EEG System time to UTC time
    start_time = start_time + timedelta(hours=7)
    end_time = end_time + timedelta(hours=7)
    
    return start_time, end_time

def load_protocol(protocol_path):
    """
    Load protocol data from a file
    param:
        protocol_path: Path to the protocol file
    return:
        protocol: Protocol data
    """
    protocol = pd.read_csv(protocol_path)
    return protocol

def load_stimulus(event_full_path, start_time, end_time):
    df = pd.read_csv(event_full_path)
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])

    df['start_time'] = pd.to_datetime(df['start_time'], unit='s', utc=True)
    df['end_time'] = pd.to_datetime(df['end_time'], unit='s', utc=True)

    # Check that patient only appeared once in the time range
    ptc_df = df[['patient_id','start_time','end_time']].groupby('patient_id',as_index=False).agg(['min', 'max'])
    ptc_df['start'] = ptc_df[('start_time', 'min')]
    ptc_df['end'] = ptc_df[('end_time', 'max')]

    ptc_df['start_str'] = ptc_df['start'].dt.strftime('%Y-%m-%d %H:%M:%S')
    ptc_df['end_str'] = ptc_df[('end_time', 'max')].dt.strftime('%Y-%m-%d %H:%M:%S')
    ptc_df = ptc_df.drop(columns=[('start_time', 'max'), ('start_time', 'min'), ('end_time', 'min'), ('end_time', 'max')])
    patient_id = ptc_df.loc[(ptc_df['start'] > start_time) & (ptc_df['end'] < end_time),'patient_id']
    if len(patient_id.index) == 1:
        patient_id = patient_id.values[0]
    else:
        raise ValueError(f"Patient has {len(patient_id.index)} stimulus activies between {start_time} and {end_time}")
    
    return df, patient_id

def trial_start_sec(row, start_time):
    return (row['start_time']-start_time).total_seconds()

def trial_end_sec(row, start_time):
    return (row['end_time']-start_time).total_seconds()

In [59]:
def detect_signal_start(raw, trial_start_sec, dc_channel):
        dc_data = raw.get_data(picks=dc_channel)
        threshold = 2 * np.std(dc_data)
        exceeds_threshold = np.where(dc_data[0] > threshold)[0]
        
        # Convert trial start time (in seconds) to sample index
        start_sample = int(trial_start_sec * raw.info['sfreq'])
        
        # Find the first exceedance after the start_sample
        signal_start_samples = exceeds_threshold[exceeds_threshold > start_sample]
        
        if signal_start_samples.size > 0:
            signal_start_sample = signal_start_samples[0]
            signal_start_time = signal_start_sample / raw.info['sfreq']
            return signal_start_sample, signal_start_time
        else:
            return None, None  # No signal detected

In [60]:
def preprocess_stim(raw, stim_csv_path, config, start_time, end_time, dc_channel):
    # Pre-processing EEG data
    try:
        ptc_df, patient_id = load_stimulus(stim_csv_path, start_time, end_time)
    except ValueError as e:
        ptc_df = pd.read_csv(stim_csv_path)
        if 'Unnamed: 0' in ptc_df.columns:
            ptc_df = ptc_df.drop(columns=['Unnamed: 0'])
        ptc_df['start_time'] = pd.to_datetime(ptc_df['start_time'], unit='s', utc=True)
        ptc_df['end_time'] = pd.to_datetime(ptc_df['end_time'], unit='s', utc=True)

        patient_id = ptc_df['patient_id'].values[0]

    patient_trial = ptc_df.loc[(ptc_df['patient_id'] == patient_id) & (ptc_df['trial_type'] == config['trial_type'])]
    patient_trial['start_sec'] = patient_trial.apply(lambda x: trial_start_sec(x, start_time), axis=1)
    patient_trial['end_sec'] = patient_trial.apply(lambda x: trial_end_sec(x, start_time), axis=1)

    signal_start_times = []
    signal_start_samples = []
    for start_sec in patient_trial['start_sec']:
        signal_start_sample, signal_start_time = detect_signal_start(raw, start_sec, dc_channel)
        signal_start_times.append(signal_start_time)
        signal_start_samples.append(signal_start_sample)

    patient_trial['signal_start_time'] = signal_start_times
    patient_trial['signal_start_sample'] = signal_start_samples
    
    return patient_trial

### Epochs

In [61]:
def get_johnsen_epochs_arr(trial_start_time, trial_end_time, signal_start_time, protocol_start_time, sfreq, verbose=False):
    events = []

    signal_start = pd.Timestamp(protocol_start_time) + pd.to_timedelta(signal_start_time, unit="s")
    
    # Extract reference epoch
    if verbose:
        print(f"Protocol start time: {protocol_start_time}")
        print(f"Patient trial start time: {trial_start_time}")
    stimulation_start_time = trial_start_time + timedelta(seconds=10)
    reference_start_time = stimulation_start_time - timedelta(seconds=2.5) #referece epoch
    reference_start_sample = int((reference_start_time - protocol_start_time).total_seconds() * sfreq)
    events.append((reference_start_sample,0,0)) 

    offset = random.uniform(0.5, 2.5)
    active_start_time = stimulation_start_time + timedelta(seconds=offset)
    active_end_time = active_start_time + timedelta(seconds=10)
    active_signal_start_time =  max(active_start_time, signal_start)
    curr_epoch_start_time = active_signal_start_time
    curr_epoch_end_time = curr_epoch_start_time + timedelta(seconds=2)
    while curr_epoch_end_time < active_end_time:
        curr_epoch_start_sample = int((curr_epoch_start_time - protocol_start_time).total_seconds() * sfreq)
        events.append((curr_epoch_start_sample,0,1))
        curr_epoch_start_time = curr_epoch_start_time + timedelta(seconds=1)
        curr_epoch_end_time = curr_epoch_start_time + timedelta(seconds=2)

    return np.array(events)


def plot_epochs(events, start_time, sampling_rate, beep_start_time):
    # Function to convert sample number to time
    def sample_to_time(sample, start_time, sampling_rate):
        return start_time + pd.to_timedelta(sample / sampling_rate, unit='s')

    # Create a time array corresponding to each event sample
    event_times = [sample_to_time(sample, start_time, sampling_rate) for sample in events[-9:, 0]]

    # Create a plot to visualize the events
    fig, ax = plt.subplots(figsize=(10, 6))

    # Plot each event
    for i, (event_time, event_type) in enumerate(zip(event_times, events[:, 2])):
        # Use event_type (0 or 1) to determine the color/label
        label = "Silence" if event_type == 0 else "Beep"
        color = "b" if event_type == 0 else "r"
        
        # Offset each event to avoid overlap in the plot
        ax.plot([event_time, event_time + pd.to_timedelta(2, unit='s')],
                [i, i], color=color, label=label if i == 0 else "")

    # Add a vertical dashed line at the stimulation time (e.g., start time)
    reference_time = beep_start_time + timedelta(seconds=10)
    ax.axvline(reference_time, color='green', linestyle='--', label="Stimulation start")

    # Set the date format for x-axis
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax.xaxis.set_major_locator(mdates.SecondLocator(interval=2))
    fig.autofmt_xdate()

    # Set labels and title
    ax.set_xlabel('Time')
    ax.set_ylabel('Epochs')
    ax.set_title('Epochs Visualization')

    # Add legend
    ax.legend()

    # Show the plot
    plt.show()

In [62]:

def create_epochs(raw, patient_trial, start_time, config):
    events = []
    for id, row in patient_trial.iterrows():
        events.extend(get_johnsen_epochs_arr(row['start_time'], row['end_time'], row['signal_start_time'], start_time, config['sfreq']))
    epochs = mne.Epochs(raw, events, tmin=0, tmax=2, baseline=None, preload=True)
    active_epochs = mne.Epochs(raw, events, event_id=1, tmin=config['tmin'], tmax=config['tmax'], picks='eeg', baseline=None, preload=True)
    reference_epochs = mne.Epochs(raw, events, event_id=0, tmin=config['tmin'], tmax=config['tmax'], picks='eeg', baseline=None, preload=True)
    return events, epochs, active_epochs, reference_epochs



### Power Computation

In [63]:
def compute_log_band_power_avg(active_psd, active_epochs, freq_bands):
    # Initialize a dictionary to hold the log-transformed average power per epoch for each band
    log_band_power_avg = {band: [] for band in freq_bands.keys()}

    # Calculate log-transformed average power in each frequency band for each epoch
    for band, (low, high) in freq_bands.items():
        band_log_power_per_epoch = []

        # Iterate through all epochs
        for psd_epoch in active_psd:
            band_power_all_channels = []

            # For each epoch, get the data and compute the power for each channel
            for i in range(len(active_epochs.ch_names)):  # Loop through all channels
                # Find indices of frequencies within the band
                band_idx = np.where((active_psd.freqs >= low) & (active_psd.freqs <= high))

                # Compute average power in the frequency band for the current channel
                band_power_value = np.mean(psd_epoch[i][band_idx])
                band_power_all_channels.append(band_power_value)

            # Compute log-transform of the average power across all channels
            avg_band_power_epoch = np.mean(band_power_all_channels)
            log_avg_band_power_epoch = np.log(avg_band_power_epoch)

            # Store the log-transformed average power for this epoch
            band_log_power_per_epoch.append(log_avg_band_power_epoch)

        # Store the log-transformed average power for each epoch in the band
        log_band_power_avg[band] = np.array(band_log_power_per_epoch)

    return log_band_power_avg


In [64]:
def plot_log_transformed_power(active_log_band_power_avg, output_path=None, verbose=False):
    # Plot the log-transformed power for each frequency band across epochs
    colors = ['b', 'orange', 'r', 'c']
    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    axs = axs.flatten()

    for i, (band, log_powers) in enumerate(active_log_band_power_avg.items()):
        axs[i].plot(range(len(log_powers)), log_powers, label=f'{band.capitalize()} Band', marker='o', color=colors[i])
        axs[i].set_title(f'Log-Transformed Average Power in {band.capitalize()} Band Across Epochs')
        axs[i].set_xlabel('Epochs')
        axs[i].set_ylabel('Log Power (µV²/Hz)')
        axs[i].grid(True)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path)
    if verbose:
        plt.show()

### Quantitative Reactivity

In [65]:
def compute_z_scores_for_bands(ref_log_band_power_avg, active_log_band_power_avg, freq_bands):
    z_score_results = {}

    # Iterate over frequency bands
    for band in freq_bands.keys():
        # Get mean and SD for the resting state in the current band
        rest_mean = np.mean(ref_log_band_power_avg[band])
        rest_std = np.std(ref_log_band_power_avg[band])

        # Compute Z-scores for the stimulation epochs
        stim_z_scores = (active_log_band_power_avg[band] - rest_mean) / rest_std

        # Identify significant increases and decreases
        significant_increase = [z > 1.96 for z in stim_z_scores]
        significant_decrease = [z < -1.96 for z in stim_z_scores]

        z_score_results[band] = {
            'z_scores': stim_z_scores,
            'significant_increase': significant_increase,
            'significant_decrease': significant_decrease
        }

    return z_score_results

In [66]:
# Plotting Z-scores for each frequency band
def plot_z_scores(z_score_results, output_path=None, verbose=False):
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    axes = axes.flatten()

    for idx, (band, results) in enumerate(z_score_results.items()):
        z_scores = results['z_scores']
        significant_increase = results['significant_increase']
        significant_decrease = results['significant_decrease']

        time = np.arange(len(z_scores))  # Assuming epochs are sequential and evenly spaced

        axes[idx].plot(time, z_scores, label='Z-scores', color='blue')
        axes[idx].axhline(1.96, color='green', linestyle='--', label='Significant threshold (+1.96)')
        axes[idx].axhline(-1.96, color='red', linestyle='--', label='Significant threshold (-1.96)')

        # Highlight significant points
        axes[idx].scatter(time[significant_increase], z_scores[significant_increase], color='green', label='Significant increase')
        axes[idx].scatter(time[significant_decrease], z_scores[significant_decrease], color='red', label='Significant decrease')

        axes[idx].set_title(f"{band} band", fontsize=15)
        axes[idx].set_xlabel("Time (epochs)", fontsize=12)
        axes[idx].set_ylabel("Z-score", fontsize=12)
        
    handles, labels = axes[3].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=2, bbox_to_anchor=(0.45, 0.03, 0.05, 0.05))
    plt.tight_layout(rect=[0, 0.9, 0, 1.02])
    plt.suptitle("Z-scores of EEG Frequency Bands After Sound Stimulation", fontsize=15, y=0.95)
    if output_path:
        plt.savefig(output_path)
    if verbose:
        plt.show()

### Config

In [67]:
config_path = '../configs/johnsen_cfg_test.yml'
with open(config_path, encoding='utf-8') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [68]:
if not os.path.exists(config['temp_eeg_dir']):
    os.makedirs(config['temp_eeg_dir'])

In [69]:
# Set the parameters
sfreq = config['sfreq']  # Sampling frequency
n_fft = sfreq*config['fft_rate']  # Length of FFT

### Part 1: Quantitative Reactivity

In [78]:
# Read catalog file
cat_df = pd.read_csv(config['catalog_path'])
cat_df['date'] = pd.to_datetime(cat_df['date'])
cat_df


for _, row in cat_df.iterrows():
    date_str = row.date.strftime('%Y%m%d')
    file_name = f"{row.patient_id}_{date_str}"

    if not str(row.patient_id) == 'CON003':
        continue

    print(f"Reading {file_name}")
    stim_csv_path = os.path.join(config['stimuli_dir'], f"{file_name}.csv")
    eeg_full_path = os.path.join(config['eeg_dir'], f"{file_name}.edf")
    if not os.path.isfile(stim_csv_path) and not os.path.isfile(eeg_full_path):
        print(f"Stimulus file {file_name} not found")
        continue

    # Create a directory for the temp results
    patient_temp_dir = os.path.join(config['temp_eeg_dir'], f"{file_name}")
    if not os.path.exists(patient_temp_dir):
        os.makedirs(patient_temp_dir)

    print(f"Processing {eeg_full_path}")
    _, raw, dc_channel = load_eeg(eeg_full_path, config)
    start_time, end_time = get_eeg_timestamps(raw)
    patient_trial = preprocess_stim(raw, stim_csv_path, config, start_time, end_time, dc_channel)
    if config['verbose']:
        print(raw.info)
        raw.plot(start=patient_trial.iloc[0]['signal_start_time'], duration=5,scalings=config['plot_scalings'])

    print(f"Creating epochs")
    events, epochs, active_epochs, ref_epochs = create_epochs(raw, patient_trial, start_time, config)
    if config['verbose']:
        plot_epochs(events, start_time, config['sfreq'], patient_trial['start_time'].values[0])

    print(f"Computing PSDs")
    # Compute PSDs for resting and stimulation epochs
    ref_psd = ref_epochs.compute_psd(method='welch', n_fft=n_fft, n_overlap=n_fft // 2, 
                                    window='hamming', fmin=config['l_freq'], fmax=config['h_freq'])
    active_psd = active_epochs.compute_psd(method='welch', n_fft=n_fft, n_overlap=n_fft // 2, 
                                    window='hamming', fmin=config['l_freq'], fmax=config['h_freq'])
    # Save intermediate PSD to a file
    ref_psd_path = os.path.join(patient_temp_dir, config['ref_psd_fname'])
    ref_psd.save(ref_psd_path, overwrite=True)
    active_psd_path = os.path.join(patient_temp_dir, config['active_psd_fname'])
    active_psd.save(active_psd_path, overwrite=True)


    print(f"Computing log-transformed average band power")
    # Calculate log-transformed average band power for resting and stimulation epochs
    ref_log_band_power_avg = compute_log_band_power_avg(ref_psd, ref_epochs, config['freq_bands'])
    active_log_band_power_avg = compute_log_band_power_avg(active_psd, active_epochs, config['freq_bands'])
    # Save log power avg to a file
    ref_log_psd_path = os.path.join(patient_temp_dir, config['ref_log_psd_fname'])
    with open(ref_log_psd_path, "w") as f:
        json.dump({band: values.tolist() for band, values in ref_log_band_power_avg.items()}, f)
    active_log_psd_path = os.path.join(patient_temp_dir, config['active_log_psd_fname'])
    with open(active_log_psd_path, "w") as f:
        json.dump({band: values.tolist() for band, values in active_log_band_power_avg.items()}, f)

    # Print the log-transformed band power for each frequency band across epochs
    if config['verbose']:
        for band, log_powers in active_log_band_power_avg.items():
            print(f"\nLog-transformed average power for {band} band across epochs:")
            print(log_powers)
    log_psd_fig_path = os.path.join(patient_temp_dir, config['log_psd_fig_fname'])
    plot_log_transformed_power(active_log_band_power_avg, log_psd_fig_path, config['verbose'])

    print(f"Quantitative Reactivity Analysis")
    # Compute Z-scores for stimulation epochs
    z_score_results = compute_z_scores_for_bands(
        ref_log_band_power_avg, active_log_band_power_avg,
        config['freq_bands'])

    # Save Z-scores to a file
    z_score_extension = os.path.splitext(config['z_score_fname'])[1]
    z_score_path = os.path.join(patient_temp_dir, config['z_score_fname'])
    
    if z_score_extension == '.npz':
        np.savez(z_score_path, **{band: values for band, values in z_score_results.items()})
    elif z_score_extension == '.json':
        z_score_result_serializable = {
            band: {
                "z_scores": values["z_scores"].tolist(),
                "significant_increase": [bool(x) for x in values["significant_increase"]],
                "significant_decrease": [bool(x) for x in values["significant_decrease"]],
            }
            for band, values in z_score_results.items()
        }
        with open("z_score_result.json", "w") as f:
            json.dump(z_score_result_serializable, f)
    
    # Print the results
    if config['verbose']:
        for band, results in z_score_results.items():
            print(f"\nBand: {band}")
            # print(f"Z-scores: {results['z_scores']}")
            print(f"Significant increases: {np.sum(results['significant_increase'])}")
            print(f"Significant decreases: {np.sum(results['significant_decrease'])}")

    # Plot the Z-scores
    z_score_fig_path = os.path.join(patient_temp_dir, config['z_score_fig_fname'])
    plot_z_scores(z_score_results, z_score_fig_path, config['verbose'])


    # break
    print("-"*20)



Reading CON003_20250114
Processing ../data/edf/CON003_20250114.edf
Extracting EDF parameters from c:\Users\nguye\OneDrive - UW\Project\EEG-project-capstone\eeg-auditory-stimulus\data\edf\CON003_20250114.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2049023  =      0.000 ...  4001.998 secs...


C:\Users\nguye\AppData\Local\Temp\ipykernel_22616\1812083879.py:6: RuntimeWarning: Omitted 5 annotation(s) that were outside data range.
  raw = mne.io.read_raw_edf(eeg_path, preload=True)


channels: ['C3', 'C4', 'O1', 'O2', 'FT9', 'FT10', 'Cz', 'F3', 'F4', 'F7', 'F8', 'Fz', 'Fp1', 'Fp2', 'Fpz', 'P3', 'P4', 'Pz', 'T7', 'T8', 'P7', 'P8', 'IO1', 'IO2', 'EMG1', 'EMG2', 'ECGL', 'ECGR', 'LAT1', 'LAT2', 'RAT1', 'RAT2', 'RESP', 'ABD', 'FLOW', 'SNORE', 'DIF5', 'DIF6', 'POS', 'DC2', 'DC3', 'DC4', 'DC5', 'DC6', 'DC7', 'DC8', 'DC9', 'DC10', 'OSAT', 'PR']
Resampling data to 256 Hz
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 70 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 70.00 Hz
- Upper transition bandwidth: 17.50 Hz (-6 dB cutoff frequency: 78.75 Hz)
- Filter length: 1691 samples (6.605 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.2s


DC channels with signals: ['DC7']
Creating epochs
Not setting metadata
54 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 54 events and 513 original time points ...
0 bad epochs dropped
Not setting metadata
48 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 48 events and 513 original time points ...
0 bad epochs dropped
Not setting metadata
6 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 6 events and 513 original time points ...
0 bad epochs dropped
Computing PSDs
Effective window size : 1.000 (s)
Effective window size : 1.000 (s)


...

Computing log-transformed average band power


C:\Users\nguye\AppData\Local\Temp\ipykernel_22616\3236571002.py:49: RuntimeWarning: This filename (../data/temp/johnsen/CON003_20250114\active_psd.fif) does not conform to MNE naming conventions. All spectrum files should end with .h5 or .hdf5
  active_psd.save(active_psd_path, overwrite=True)


Quantitative Reactivity Analysis


C:\Users\nguye\AppData\Local\Temp\ipykernel_22616\842597429.py:27: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout(rect=[0, 0.9, 0, 1.02])


--------------------


In [86]:
patient_trial

,patient_id,date,trial_type,sentences,start_time,end_time,duration,start_sec,end_sec,signal_start_time,signal_start_sample
22,CON003,1/14/2025,beep,[],2025-01-14 23:53:18+00:00,2025-01-14 23:53:48+00:00,30.362823,1441.0,1471.0,1452.089844,371735
35,CON003,1/14/2025,beep,[],2025-01-15 00:03:41+00:00,2025-01-15 00:04:11+00:00,30.220063,2064.0,2094.0,2075.292969,531275
53,CON003,1/14/2025,beep,[],2025-01-15 00:15:29+00:00,2025-01-15 00:16:00+00:00,30.213353,2772.0,2803.0,2783.750000,712640
62,CON003,1/14/2025,beep,[],2025-01-15 00:18:18+00:00,2025-01-15 00:18:49+00:00,30.214854,2941.0,2972.0,2952.558594,755855
71,CON003,1/14/2025,beep,[],2025-01-15 00:21:08+00:00,2025-01-15 00:21:38+00:00,30.211435,3111.0,3141.0,3121.976562,799226
79,CON003,1/14/2025,beep,[],2025-01-15 00:23:41+00:00,2025-01-15 00:24:11+00:00,30.233144,3264.0,3294.0,3274.824219,838355


### Load Intermediate Files

In [ ]:
from mne.time_frequency import read_spectrum
patient_temp_dir = os.path.join(config['temp_eeg_dir'], f"{file_name}")

ref_psd_path = os.path.join(patient_temp_dir, config['ref_psd_fname'])
active_psd_path = os.path.join(patient_temp_dir, config['active_psd_fname'])

ref_psd = read_spectrum(ref_psd_path)
active_psd = read_spectrum(active_psd_path)


In [ ]:
ref_log_psd_path = os.path.join(patient_temp_dir, config['ref_log_psd_fname'])
active_log_psd_path = os.path.join(patient_temp_dir, config['active_log_psd_fname'])

with open(ref_log_psd_path, "r") as f:
    ref_log_band_power_avg = {band: np.array(values) for band, values in json.load(f).items()}
with open(active_log_psd_path, "r") as f:
    active_log_psd_path = {band: np.array(values) for band, values in json.load(f).items()}

In [ ]:
z_score_extension = os.path.splitext(config['z_score_fname'])[1]
z_score_path = os.path.join(patient_temp_dir, config['z_score_fname'])

if z_score_extension == '.npz':
    data = np.load(z_score_path, allow_pickle=True)
    z_score_result = {band: data[band].item() for band in data.files}
elif z_score_extension == '.json':
    with open(z_score_path, "r") as f:
        z_score_result = json.load(f)
    # Convert lists back to NumPy arrays
    z_score_result = {
        band: {
            "z_scores": np.array(values["z_scores"]),
            "significant_increase": np.array(values["significant_increase"]),
            "significant_decrease": np.array(values["significant_decrease"]),
        }
        for band, values in z_score_result.items()
    }

